# 03 — Evaluation Traces

Evaluates the NRCS Conservation Program Navigator and produces the LangSmith traces plus the written performance commentary.

**What this notebook does**
- Runs the agent over a 10 example dataset (7 in scope, 3 out of scope) with tracing on, so every run is captured in LangSmith.
- Scores each run with a mix of deterministic evaluators and an LLM judge (a pass/fail checklist, not a 1-5 scale).
- Runs the **same dataset through two models** (premier `gpt-4o` vs. cheaper `gpt-4o-mini`) for a side by side comparison.
- Surfaces the ROI numbers (tokens, latency, cost) per model.
- Frames the human in the loop step: hand annotation in a LangSmith queue to validate the judge.

The logic lives in `src/nrcs_navigator/evaluation/` (`datasets.py`, `judge.py`, `run_traces.py`); this notebook stays thin and calls into it.

## Setup

Import the evaluation modules and confirm the model legs and LangSmith tracing are configured. `config` loads `.env` on import, so `LANGCHAIN_API_KEY` and `LANGCHAIN_TRACING_V2` are populated for the traces.

In [1]:
%load_ext autoreload
%autoreload 2

import os

from nrcs_navigator import config
from nrcs_navigator.evaluation import datasets, judge, run_traces

print("premier model:", config.PREMIER_MODEL)
print("cheap model:  ", config.CHEAP_MODEL)
print("tracing on:   ", os.environ.get("LANGCHAIN_TRACING_V2"))
print("project:      ", os.environ.get("LANGCHAIN_PROJECT"))
print("evaluators:   ", [e.__name__ for e in judge.EVALUATORS])

premier model: gpt-4o
cheap model:   gpt-4o-mini
tracing on:    true
project:       nrcs_navigator
evaluators:    ['scope_adherence', 'tool_trajectory', 'tools_succeeded', 'program_match', 'llm_judge']


## Dataset

The eval examples are defined in `datasets.py` (source of truth, in git) and pushed to a named LangSmith dataset so runs are repeatable. Each example carries `in_scope`, `expected_programs`, `expected_tools`, and a prose `expectations` rubric the judge grades against.

Push is idempotent (it recreates the dataset), so re-running this cell is safe.

In [2]:
dataset_id = datasets.push_to_langsmith()

examples = datasets.EVAL_EXAMPLES
in_scope = sum(1 for e in examples if e["in_scope"])
print(f"dataset '{datasets.DATASET_NAME}' ({dataset_id})")
print(f"{len(examples)} examples: {in_scope} in scope, {len(examples) - in_scope} out of scope")

dataset 'nrcs-navigator-eval' (81300084-d910-4ab9-8439-bb430d4e082f)
10 examples: 6 in scope, 4 out of scope


## Run the comparison: premier vs. cheaper

`run_comparison` runs the **same dataset** through both models, applying every evaluator to each run. This is the multiple model requirement: the only thing that changes between the two legs is the agent's LLM (swapped via `model_name`), not the tools or the graph.

Runs serially because `program_availability` drives a headless browser. Each leg prints a LangSmith experiment URL — open them to inspect individual traces.

In [3]:
results = run_traces.run_comparison()
results

View the evaluation results for experiment: 'nrcs-gpt-4o-4b02b052' at:
https://smith.langchain.com/o/67f30057-27a7-4c02-922d-cf687d68029b/datasets/81300084-d910-4ab9-8439-bb430d4e082f/compare?selectedSessions=f3e4da8a-da40-422d-a6dc-2ea0da86735a




0it [00:00, ?it/s]

Estimating payments...


Error running evaluator <DynamicRunEvaluator scope_adherence> on run 095ca70f-f353-4f9b-8ca4-dc611b6fc1b0: ValueError('Expected a non-empty dict, str, bool, int, float, list, EvaluationResult, or EvaluationResults. Got None')
Traceback (most recent call last):
  File "/Users/seanmcgowan/Library/Caches/pypoetry/virtualenvs/nrcs-navigator--lsF73BZ-py3.11/lib/python3.11/site-packages/langsmith/evaluation/_runner.py", line 1404, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(
                         ^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/seanmcgowan/Library/Caches/pypoetry/virtualenvs/nrcs-navigator--lsF73BZ-py3.11/lib/python3.11/site-packages/langsmith/evaluation/evaluator.py", line 336, in evaluate_run
    return self._format_result(result, source_run_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/seanmcgowan/Library/Caches/pypoetry/virtualenvs/nrcs-navigator--lsF73BZ-py3.11/lib/python3.11/site-packages/langsmith/evaluation/evaluator.py", li

Checking available programs...
Screening eligibility...
Checking available programs...
Screening eligibility...
Estimating payments...
Matching practices...
Screening eligibility...
Estimating payments...
Checking available programs...
Matching practices...


KeyboardInterrupt: 

## Scorecard

Mean of each feedback metric per model. Two metric families:
- **Pass/fail (binary):** `scope_adherence`, `tools_succeeded`, and the judge criteria (`no_fabrication`, `claims_cited`, `addresses_question`, `correct_redirect`) — read as pass rates. `correct_redirect` applies to out of scope examples only, so read it against the out of scope cases rather than comparing it to the in scope criteria.
- **Coverage (0-1):** `program_match`, `tool_trajectory` — read as mean coverage.

Not applicable cells (e.g. tool checks on an out of scope decline) score null and drop out of the averages.

In [5]:
scorecard = run_traces.summarize_results(results)
scorecard.round(3)

,gpt-4o,gpt-4o-mini
scope_adherence,1.000,0.900
tool_trajectory,0.952,0.857
tools_succeeded,1.000,1.000
program_match,1.000,1.000
correct_redirect,1.000,1.000
no_fabrication,0.857,0.714
claims_cited,0.571,0.286
addresses_question,0.571,0.714
acep_no_figure,1.000,1.000
avg_total_tokens,8223.400,7576.100


## ROI inputs: cost vs. effectiveness

Per model averages for latency and token usage, pulled from the experiment traces. Combined with the scorecard above, these are the raw numbers behind the cost vs. effectiveness argument: how much quality the premier model buys for its extra cost.

In [7]:
# Latency, tokens, and dollar cost per model, read from the LangSmith run roots
# (the agent invocations). Pair with the scorecard above for cost vs. effectiveness.
run_traces.roi_table(results).round(5)

,gpt-4o,gpt-4o-mini
avg_latency_s,10.83938,9.93176
avg_total_tokens,8223.40000,7576.10000
avg_cost_usd,0.01512,0.00092
total_cost_usd,0.15116,0.00916


## RAG retrieval evaluation (component eval)

Separate from the agent harness above. This tests the `eligibility_screener` retriever **directly** (`vectorstore.similarity_search`), not through the agent, so a poor score is attributable to retrieval rather than to the model's reasoning.

Each example is a question paired with the single gold CFR section that answers it. The gold citations come from the section **headings** (true structure), and the questions are deliberately **paraphrased to avoid the heading vocabulary**, so this measures semantic retrieval, not keyword overlap. Metrics:
- **hit@k** — is the gold section in the top k retrieved (recall; one gold per query).
- **MRR** — mean reciprocal rank of the gold section (rewards ranking it higher).

This is a deterministic, offline computation, so it lives in plain Python (`evaluation/rag_eval.py`) rather than LangSmith.

In [1]:
import pandas as pd

from nrcs_navigator.evaluation import rag_eval

# Summary: hit@k and MRR, unfiltered vs. with the program metadata filter.
summary = pd.concat([
    rag_eval.evaluate(use_program_filter=False),
    rag_eval.evaluate(use_program_filter=True),
])
display(summary.round(3))

# Row by row: where the gold section ranked, and the misses.
rag_eval.per_example(k=5)

,hit@1,hit@3,hit@5,MRR,n
unfiltered,0.417,0.75,0.75,0.583,12.0
filtered,0.417,0.75,0.75,0.583,12.0


,question,program,gold,rank,hit,reciprocal_rank
0,Does my client's operation qualify to take par...,CSP,7 CFR 1470.6,2.0,True,0.5
1,How is the amount of money a Conservation Stew...,CSP,7 CFR 1470.24,1.0,True,1.0
2,When joining the Conservation Stewardship Prog...,CSP,7 CFR 1470.22,2.0,True,0.5
3,After a Conservation Stewardship Program agree...,CSP,7 CFR 1470.26,1.0,True,1.0
4,"Under EQIP, how does NRCS set the dollar amoun...",EQIP,7 CFR 1466.23,2.0,True,0.5
5,What countrywide resource concerns is EQIP mea...,EQIP,7 CFR 1466.4,1.0,True,1.0
6,"When entering an EQIP contract, what document ...",EQIP,7 CFR 1466.7,2.0,True,0.5
7,"Through EQIP, how can an organization receive ...",EQIP,7 CFR 1466.32,NaN,False,0.0
8,If a farmer sells the development rights on th...,ACEP,7 CFR 1468.24,1.0,True,1.0
9,"Under ACEP, can a partner buy land, place a pe...",ACEP,7 CFR 1468.27,NaN,False,0.0


## Human in the loop: validating the judge

An LLM judge is itself a model that can be wrong, so before trusting its premier vs. cheaper verdict it is validated against human labels.

**Process**
1. In LangSmith, send a subset of these runs to an **annotation queue** (include runs from *both* models, especially ones where they disagree, so the judge is tested on good answers and bad ones).
2. Hand label the same pass/fail criteria the judge uses.
3. Compute agreement (judge vs. human) per criterion. High agreement -> trust the judge at scale; low agreement -> sharpen that criterion's definition in `judge.CRITERIA` and re-judge.

_Annotation findings and the judge-human agreement go here once the queue is labeled._

In [ ]:
# Run this AFTER annotating a subset of the runs above in a LangSmith annotation
# queue (label the same pass/fail criteria the judge uses). It lines up each
# human label against the judge's score on the same run and criterion, and
# reports per criterion agreement plus Cohen's kappa (chance corrected).
#
# Empty until annotations exist. High agreement -> trust the judge; low
# agreement on a criterion -> sharpen its definition in judge.CRITERIA, re-judge,
# and re-check here.
experiment_names = [getattr(exp, "experiment_name", None) for exp in results.values()]
agreement = run_traces.judge_human_agreement([n for n in experiment_names if n])
agreement.round(3)

## Performance commentary

_Written analysis:_
- How the agent performed overall (in scope research, out of scope declines).
- What the judge revealed (where the cheaper model fabricated or skipped citations / tools).
- Premier vs. cheaper trade off read against the ROI table.
- How the human was involved (annotation + judge calibration above).